# NB05 — Triangulation → Point Cloud（收成課）
到而家我哋有：絕對相位圖（nb03）+ 完整標定（nb04）。
每個相機 pixel 給出一條**射線**，絕對相位給出投影儀嘅一個**平面**——
射線 ∩ 平面 = 3D 點。C++ 叫呢步 `reverseCamera`：逐 pixel 解 3×3 線性系統。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## 1. 解碼真實掃描數據

In [2]:
import cv2
from sl_edu import decode, oracle, reconstruct

imgs = oracle.load_shift_graycode(ROOT)
wrapped, conf, absolute = oracle.decode_all(imgs)
print(f'absolute phase decoded: {np.count_nonzero(absolute):,} valid pixels')

absolute phase decoded: 1,310,720 valid pixels


## 2. 載入標定 + 組投影矩陣
$$P_L = K_{cam}[I|0],\quad P_R = K_{proj}[R^T\ |\ -R^T T]$$
（世界座標 = 相機座標；caliInfo 嘅 Rlp/Tlp 係 camera→projector，所以用逆。）

In [3]:
fs = cv2.FileStorage(str(DATA / 'monocularCamera' / 'caliInfo.yml'),
                     cv2.FILE_STORAGE_READ)
M1, D1 = fs.getNode('M1').mat(), fs.getNode('D1').mat()
M4, D4 = fs.getNode('M4').mat(), fs.getNode('D4').mat()
Rlp, Tlp = fs.getNode('Rlp').mat(), fs.getNode('Tlp').mat()
fs.release()

# 先將相位圖去畸變（等效於喺 normalized 相機模型下工作）
absolute_u = cv2.undistort(absolute, M1, D1)

PL = M1 @ np.hstack([np.eye(3), np.zeros((3, 1))])
PR = M4 @ np.hstack([Rlp, Tlp])   # P_proj-frame = Rlp @ P_cam + Tlp
pitch = 1920 / 32                  # projector pixels per period
print('PL:'); print(np.round(PL, 1))
print('pitch =', pitch, 'px/period')

PL:
[[1.7541e+03 0.0000e+00 6.4540e+02 0.0000e+00]
 [0.0000e+00 1.7529e+03 5.2950e+02 0.0000e+00]
 [0.0000e+00 0.0000e+00 1.0000e+00 0.0000e+00]]
pitch = 60.0 px/period


## 3. 逐 pixel 深度恢復
教學版直接 double loop（C++ 都係咁，不過 parallel_for）。
1280×1024 全圖會行幾分鐘——教學上先 crop 一個 ROI 示範，全圖留俾你行。

In [4]:
# ROI: 中央 400x400，快啲睇到結果
r0, c0, sz = 312, 440, 400
roi = absolute_u[r0:r0+sz, c0:c0+sz]

# 平移投影矩陣嘅主點，等效於 crop
PL_roi = PL.copy(); PL_roi[0, 2] -= c0; PL_roi[1, 2] -= r0

depth = reconstruct.depth_from_phase(roi, PL_roi, PR, pitch,
                                     min_depth=100, max_depth=2000)
valid = depth[depth > 0]
print(f'{valid.size:,} valid depths | median {np.median(valid):.0f} mm | '
      f'range {valid.min():.0f}..{valid.max():.0f} mm')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(absolute > 0, cmap='gray'); plt.title('valid mask')
plt.subplot(1, 2, 2)
plt.imshow(depth, cmap='turbo'); plt.colorbar(label='depth (mm)')
plt.title('depth map (ROI)'); plt.show()

159,635 valid depths | median 1054 mm | range 194..1996 mm


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82705/1332737306.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('depth map (ROI)'); plt.show()


## 4. 點雲輸出

In [5]:
from sl_edu import reconstruct as rc

ys, xs = np.nonzero(depth)
pts_cam = np.column_stack([xs, ys]).astype(float)
z = depth[ys, xs]
# back-project with intrinsics (ROI-adjusted)
X = (xs + c0 - M1[0, 2]) * z / M1[0, 0]
Y = (ys + r0 - M1[1, 2]) * z / M1[1, 1]
cloud = np.column_stack([X, Y, z])

out = rc.save_ply(ROOT / 'edu' / 'output_scan.ply', cloud)
print(f'saved {len(cloud):,} points -> {out}')
print('open it in MeshLab / CloudCompare / open3d to inspect!')

saved 159,635 points -> /private/tmp/slmaster-fork/edu/output_scan.ply
open it in MeshLab / CloudCompare / open3d to inspect!


## 你啱啱做咗咩
9 張灰階相 → 相位解碼 → 逐 pixel 三角化 → **真實場景嘅 3D 點雲**。
成條 SLMaster pipeline 嘅概念核心就係咁多。C++ 版本做嘅係同一件事，
只係快 100 倍（SIMD + parallel_for + 可選 CUDA）加埋硬件控制。

下一課：用真。webcam + 芒行一次 live scan（或者繼續 virtual）。